# Install R

In [1]:
%load_ext rpy2.ipython


In [2]:
%%R
install.packages("nhanesA", repos="https://cloud.r-project.org")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependency ‘plyr’

trying URL 'https://cloud.r-project.org/src/contrib/plyr_1.8.9.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/nhanesA_1.4.1.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpQ3LGzQ/downloaded_packages’


# Connect Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


# Read in Restricted Patient ID's

In [4]:
%%R

library(dplyr)

# read restricted patient SEQN file
post_2_restricted_patients <- read.csv(
  "/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/Post_2-Restricted_Patients.csv"
)

head(post_2_restricted_patients)

   SEQN
1 62174
2 62177
3 62178
4 62179
5 62191
6 62199



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



# Add the Covariates


In [5]:
%%R

library(nhanesA)
library(dplyr)

cat("Starting full NHANES time-fixed covariate table creation...\n\n")

download_table <- function(prefix, suffix, cycle_label, vars) {

  table_name <- paste0(prefix, "_", suffix)
  cat("Downloading:", table_name, "\n")

  df <- nhanes(table_name)

  cat(table_name, "rows:", nrow(df), "\n")
  cat(table_name, "cols:", ncol(df), "\n")

  df_clean <- df %>%
    select(SEQN, all_of(vars)) %>%
    mutate(cycle = cycle_label)

  cat("Selected rows:", nrow(df_clean), "\n")
  cat("Selected cols:", ncol(df_clean), "\n\n")

  return(df_clean)
}

# ----------------------------
# 1. Download G cycle, 2011-2012
# ----------------------------

demo_g <- download_table("DEMO", "G", "2011-2012",
                         c("RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "DMDMARTL"))

bmx_g <- download_table("BMX", "G", "2011-2012",
                        c("BMXBMI"))

smq_g <- download_table("SMQ", "G", "2011-2012",
                        c("SMQ020", "SMQ040"))

alq_g <- download_table("ALQ", "G", "2011-2012",
                        c("ALQ101"))

bpq_g <- download_table("BPQ", "G", "2011-2012",
                        c("BPQ020"))

diq_g <- download_table("DIQ", "G", "2011-2012",
                        c("DIQ010"))

mcq_g <- download_table("MCQ", "G", "2011-2012",
                        c("MCQ160E", "MCQ160F", "MCQ220"))

huq_g <- download_table("HUQ", "G", "2011-2012",
                        c("HUQ010"))

# ----------------------------
# 2. Download H cycle, 2013-2014
# ----------------------------

demo_h <- download_table("DEMO", "H", "2013-2014",
                         c("RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "DMDMARTL"))

bmx_h <- download_table("BMX", "H", "2013-2014",
                        c("BMXBMI"))

smq_h <- download_table("SMQ", "H", "2013-2014",
                        c("SMQ020", "SMQ040"))

alq_h <- download_table("ALQ", "H", "2013-2014",
                        c("ALQ101"))

bpq_h <- download_table("BPQ", "H", "2013-2014",
                        c("BPQ020"))

diq_h <- download_table("DIQ", "H", "2013-2014",
                        c("DIQ010"))

mcq_h <- download_table("MCQ", "H", "2013-2014",
                        c("MCQ160E", "MCQ160F", "MCQ220"))

huq_h <- download_table("HUQ", "H", "2013-2014",
                        c("HUQ010"))

# ----------------------------
# 3. Bind each table across cycles
# ----------------------------

cat("Combining cycles within each table...\n\n")

demo_raw <- bind_rows(demo_g, demo_h)
bmx_raw  <- bind_rows(bmx_g, bmx_h)
smq_raw  <- bind_rows(smq_g, smq_h)
alq_raw  <- bind_rows(alq_g, alq_h)
bpq_raw  <- bind_rows(bpq_g, bpq_h)
diq_raw  <- bind_rows(diq_g, diq_h)
mcq_raw  <- bind_rows(mcq_g, mcq_h)
huq_raw  <- bind_rows(huq_g, huq_h)

cat("DEMO rows:", nrow(demo_raw), "\n")
cat("BMX rows:", nrow(bmx_raw), "\n")
cat("SMQ rows:", nrow(smq_raw), "\n")
cat("ALQ rows:", nrow(alq_raw), "\n")
cat("BPQ rows:", nrow(bpq_raw), "\n")
cat("DIQ rows:", nrow(diq_raw), "\n")
cat("MCQ rows:", nrow(mcq_raw), "\n")
cat("HUQ rows:", nrow(huq_raw), "\n\n")

# ----------------------------
# 4. Merge all tables
# ----------------------------

cat("Merging all covariate tables by SEQN and cycle...\n\n")

nhanes_covariates_raw <- demo_raw %>%
  left_join(bmx_raw, by = c("SEQN", "cycle")) %>%
  left_join(smq_raw, by = c("SEQN", "cycle")) %>%
  left_join(alq_raw, by = c("SEQN", "cycle")) %>%
  left_join(bpq_raw, by = c("SEQN", "cycle")) %>%
  left_join(diq_raw, by = c("SEQN", "cycle")) %>%
  left_join(mcq_raw, by = c("SEQN", "cycle")) %>%
  left_join(huq_raw, by = c("SEQN", "cycle"))

cat("Merged rows:", nrow(nhanes_covariates_raw), "\n")
cat("Merged cols:", ncol(nhanes_covariates_raw), "\n\n")

# ----------------------------
# 5. Inspect raw values
# ----------------------------

cat("Variable classes in merged raw table:\n")
print(sapply(nhanes_covariates_raw, class))

cat("\nUnique HUQ010 values:\n")
print(unique(nhanes_covariates_raw$HUQ010))

cat("\nUnique DIQ010 values:\n")
print(unique(nhanes_covariates_raw$DIQ010))

cat("\nUnique smoking values:\n")
print(unique(nhanes_covariates_raw$SMQ020))
print(unique(nhanes_covariates_raw$SMQ040))

# ----------------------------
# 6. Create cleaned time-fixed covariate table
# ----------------------------

cat("\nCreating cleaned time-fixed covariate table...\n\n")

time_fixed_covariates <- nhanes_covariates_raw %>%
  transmute(
    SEQN,
    cycle,

    age = RIDAGEYR,

    sex = as.character(RIAGENDR),

    race_ethnicity = as.character(RIDRETH3),

    education_raw = as.character(DMDEDUC2),

    marital_status_raw = as.character(DMDMARTL),

    bmi = BMXBMI,

    smoked_100_cigarettes_raw = as.character(SMQ020),

    current_smoking_raw = as.character(SMQ040),

    alcohol_raw = as.character(ALQ101),

    hypertension_raw = as.character(BPQ020),

    diabetes_raw = as.character(DIQ010),

    heart_attack_raw = as.character(MCQ160E),

    stroke_raw = as.character(MCQ160F),

    cancer_raw = as.character(MCQ220),

    self_rated_health_raw = as.character(HUQ010)
  ) %>%
  mutate(

    education = case_when(
      education_raw == "High school graduate/GED or equivalent" ~ "High school/GED",
      education_raw == "Some college or AA degree" ~ "Some college/AA",
      education_raw == "College graduate or above" ~ "College graduate+",
      education_raw == "9-11th grade (Includes 12th grade with no diploma)" ~ "9-11th grade",
      education_raw == "Less than 9th grade" ~ "<9th grade",
      education_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ education_raw
    ),

    marital_status = case_when(
      marital_status_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ marital_status_raw
    ),

    smoking_status = case_when(
      smoked_100_cigarettes_raw == "No" ~ "Never",
      smoked_100_cigarettes_raw == "Yes" &
        current_smoking_raw == "Not at all" ~ "Former",
      smoked_100_cigarettes_raw == "Yes" &
        current_smoking_raw %in% c("Every day", "Some days") ~ "Current",
      TRUE ~ NA_character_
    ),

    alcohol_use = case_when(
      alcohol_raw == "Yes" ~ "Yes",
      alcohol_raw == "No" ~ "No",
      alcohol_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ NA_character_
    ),

    hypertension = case_when(
      hypertension_raw == "Yes" ~ "Yes",
      hypertension_raw == "No" ~ "No",
      hypertension_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ NA_character_
    ),

    diabetes = case_when(
      diabetes_raw == "Yes" ~ "Yes",
      diabetes_raw == "No" ~ "No",
      diabetes_raw == "Borderline" ~ "No",
      diabetes_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ NA_character_
    ),

    heart_attack = case_when(
      heart_attack_raw == "Yes" ~ "Yes",
      heart_attack_raw == "No" ~ "No",
      heart_attack_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ NA_character_
    ),

    stroke = case_when(
      stroke_raw == "Yes" ~ "Yes",
      stroke_raw == "No" ~ "No",
      stroke_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ NA_character_
    ),

    cancer = case_when(
      cancer_raw == "Yes" ~ "Yes",
      cancer_raw == "No" ~ "No",
      cancer_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ NA_character_
    ),

    self_rated_health = case_when(
      self_rated_health_raw == "Excellent," ~ "Excellent",
      self_rated_health_raw == "Very good," ~ "Very good",
      self_rated_health_raw == "Good," ~ "Good",
      self_rated_health_raw == "Fair, or" ~ "Fair",
      self_rated_health_raw == "Poor?" ~ "Poor",
      self_rated_health_raw %in% c("Refused", "Don't Know", "Don't know") ~ NA_character_,
      TRUE ~ NA_character_
    )
  ) %>%
  select(
    SEQN,
    cycle,
    age,
    sex,
    race_ethnicity,
    education,
    marital_status,
    bmi,
    smoking_status,
    alcohol_use,
    hypertension,
    diabetes,
    heart_attack,
    stroke,
    cancer,
    self_rated_health
  )

cat("Finished cleaned table.\n")
cat("Rows:", nrow(time_fixed_covariates), "\n")
cat("Cols:", ncol(time_fixed_covariates), "\n\n")

# ----------------------------
# 7. Optional age restriction
# ----------------------------

cat("Creating age 50-80 analytic version...\n\n")

time_fixed_covariates_50_80 <- time_fixed_covariates %>%
  filter(age >= 50, age <= 80)

cat("Rows before age filter:", nrow(time_fixed_covariates), "\n")
cat("Rows after age 50-80 filter:", nrow(time_fixed_covariates_50_80), "\n\n")

# ----------------------------
# 8. Print first rows of final product
# ----------------------------

cat("First 10 rows of final cleaned time-fixed covariate table:\n\n")
print(as.data.frame(head(time_fixed_covariates, 10)))

cat("\nFirst 10 rows of age 50-80 analytic table:\n\n")
print(as.data.frame(head(time_fixed_covariates_50_80, 10)))

# ----------------------------
# 9. Missingness check
# ----------------------------

cat("\nMissingness count by variable, full table:\n\n")

missingness_table <- data.frame(
  variable = names(time_fixed_covariates),
  n_missing = sapply(time_fixed_covariates, function(x) sum(is.na(x))),
  percent_missing = round(
    100 * sapply(time_fixed_covariates, function(x) mean(is.na(x))),
    1
  )
)

print(as.data.frame(missingness_table))

# ----------------------------
# 10. Simple baseline summary table, age 50-80
# ----------------------------

cat("\nCreating simple baseline summary table for age 50-80...\n\n")

make_cat_summary <- function(data, var_name) {

  out <- data %>%
    filter(!is.na(.data[[var_name]])) %>%
    count(!!sym(var_name), name = "n") %>%
    mutate(
      variable = var_name,
      level = as.character(!!sym(var_name)),
      percent = round(100 * n / sum(n), 1),
      summary = paste0(n, " (", percent, "%)")
    ) %>%
    select(variable, level, summary)

  return(as.data.frame(out))
}

age_summary <- data.frame(
  variable = "age",
  level = "Mean (SD)",
  summary = paste0(
    round(mean(time_fixed_covariates_50_80$age, na.rm = TRUE), 1),
    " (",
    round(sd(time_fixed_covariates_50_80$age, na.rm = TRUE), 1),
    ")"
  )
)

bmi_summary <- data.frame(
  variable = "bmi",
  level = "Mean (SD)",
  summary = paste0(
    round(mean(time_fixed_covariates_50_80$bmi, na.rm = TRUE), 1),
    " (",
    round(sd(time_fixed_covariates_50_80$bmi, na.rm = TRUE), 1),
    ")"
  )
)

baseline_table <- bind_rows(
  age_summary,
  bmi_summary,
  make_cat_summary(time_fixed_covariates_50_80, "sex"),
  make_cat_summary(time_fixed_covariates_50_80, "race_ethnicity"),
  make_cat_summary(time_fixed_covariates_50_80, "education"),
  make_cat_summary(time_fixed_covariates_50_80, "marital_status"),
  make_cat_summary(time_fixed_covariates_50_80, "smoking_status"),
  make_cat_summary(time_fixed_covariates_50_80, "alcohol_use"),
  make_cat_summary(time_fixed_covariates_50_80, "hypertension"),
  make_cat_summary(time_fixed_covariates_50_80, "diabetes"),
  make_cat_summary(time_fixed_covariates_50_80, "heart_attack"),
  make_cat_summary(time_fixed_covariates_50_80, "stroke"),
  make_cat_summary(time_fixed_covariates_50_80, "cancer"),
  make_cat_summary(time_fixed_covariates_50_80, "self_rated_health")
)

cat("\nFinal baseline table, age 50-80:\n\n")
print(as.data.frame(baseline_table))

Starting full NHANES time-fixed covariate table creation...

Downloading: DEMO_G 
DEMO_G rows: 9756 
DEMO_G cols: 48 
Selected rows: 9756 
Selected cols: 7 

Downloading: BMX_G 
BMX_G rows: 9338 
BMX_G cols: 26 
Selected rows: 9338 
Selected cols: 3 

Downloading: SMQ_G 
SMQ_G rows: 6790 
SMQ_G cols: 30 
Selected rows: 6790 
Selected cols: 4 

Downloading: ALQ_G 
ALQ_G rows: 5615 
ALQ_G cols: 10 
Selected rows: 5615 
Selected cols: 3 

Downloading: BPQ_G 
BPQ_G rows: 6175 
BPQ_G cols: 15 
Selected rows: 6175 
Selected cols: 3 

Downloading: DIQ_G 
DIQ_G rows: 9364 
DIQ_G cols: 53 
Selected rows: 9364 
Selected cols: 3 

Downloading: MCQ_G 
MCQ_G rows: 9364 
MCQ_G cols: 92 
Selected rows: 9364 
Selected cols: 5 

Downloading: HUQ_G 
HUQ_G rows: 9756 
HUQ_G cols: 10 
Selected rows: 9756 
Selected cols: 3 

Downloading: DEMO_H 
DEMO_H rows: 10175 
DEMO_H cols: 47 
Selected rows: 10175 
Selected cols: 7 

Downloading: BMX_H 
BMX_H rows: 9813 
BMX_H cols: 26 
Selected rows: 9813 
Selected c

In [6]:
%%R

library(dplyr)

# ----------------------------
# Restrict covariate table
# to analytic patients only
# ----------------------------

time_fixed_covariates_restricted <- time_fixed_covariates %>%
  semi_join(
    post_2_restricted_patients,
    by = "SEQN"
  )

cat("Restricted covariate table rows:",
    nrow(time_fixed_covariates_restricted), "\n")

cat("Restricted unique patients:",
    n_distinct(time_fixed_covariates_restricted$SEQN), "\n\n")

print(head(time_fixed_covariates_restricted))

# ----------------------------
# Baseline table helper
# ----------------------------

make_cat_summary <- function(data, var_name) {

  out <- data %>%
    filter(!is.na(.data[[var_name]])) %>%
    count(!!sym(var_name), name = "n") %>%
    mutate(
      variable = var_name,
      level = as.character(!!sym(var_name)),
      percent = round(100 * n / sum(n), 1),
      summary = paste0(n, " (", percent, "%)")
    ) %>%
    select(variable, level, summary)

  return(as.data.frame(out))
}

# ----------------------------
# Continuous summaries
# ----------------------------

age_summary <- data.frame(
  variable = "age",
  level = "Mean (SD)",
  summary = paste0(
    round(mean(time_fixed_covariates_restricted$age, na.rm = TRUE), 1),
    " (",
    round(sd(time_fixed_covariates_restricted$age, na.rm = TRUE), 1),
    ")"
  )
)

bmi_summary <- data.frame(
  variable = "bmi",
  level = "Mean (SD)",
  summary = paste0(
    round(mean(time_fixed_covariates_restricted$bmi, na.rm = TRUE), 1),
    " (",
    round(sd(time_fixed_covariates_restricted$bmi, na.rm = TRUE), 1),
    ")"
  )
)

# ----------------------------
# Final baseline table
# ----------------------------

baseline_table <- bind_rows(

  age_summary,

  bmi_summary,

  make_cat_summary(time_fixed_covariates_restricted, "sex"),

  make_cat_summary(time_fixed_covariates_restricted, "race_ethnicity"),

  make_cat_summary(time_fixed_covariates_restricted, "education"),

  make_cat_summary(time_fixed_covariates_restricted, "marital_status"),

  make_cat_summary(time_fixed_covariates_restricted, "smoking_status"),

  make_cat_summary(time_fixed_covariates_restricted, "alcohol_use"),

  make_cat_summary(time_fixed_covariates_restricted, "hypertension"),

  make_cat_summary(time_fixed_covariates_restricted, "diabetes"),

  make_cat_summary(time_fixed_covariates_restricted, "heart_attack"),

  make_cat_summary(time_fixed_covariates_restricted, "stroke"),

  make_cat_summary(time_fixed_covariates_restricted, "cancer"),

  make_cat_summary(time_fixed_covariates_restricted, "self_rated_health")
)

# ----------------------------
# Print final table
# ----------------------------

cat("\nRestricted patient baseline table:\n\n")

print(as.data.frame(baseline_table))

Restricted covariate table rows: 4625 
Restricted unique patients: 4625 

   SEQN     cycle age  sex     race_ethnicity         education
1 62174 2011-2012  80 Male Non-Hispanic White College graduate+
2 62177 2011-2012  51 Male Non-Hispanic Asian   High school/GED
3 62178 2011-2012  80 Male Non-Hispanic White   High school/GED
4 62179 2011-2012  55 Male Non-Hispanic Asian College graduate+
5 62191 2011-2012  70 Male Non-Hispanic Black   High school/GED
6 62199 2011-2012  57 Male Non-Hispanic White College graduate+
       marital_status  bmi smoking_status alcohol_use hypertension diabetes
1             Married 33.9          Never         Yes          Yes       No
2             Married 20.1        Current        <NA>           No       No
3             Widowed 28.5          Never          No           No       No
4             Married 27.6          Never          No           No       No
5            Divorced   NA         Former         Yes           No       No
6 Living with partner 

# Add Mortality

In [7]:
%%R

library(dplyr)
library(readr)

cat("Starting restricted all-cause mortality linkage...\n\n")

# ----------------------------
# 1. Read restricted patient IDs
# ----------------------------

post_2_restricted_patients <- read.csv(
  "/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/Post_2-Restricted_Patients.csv"
) %>%
  distinct(SEQN)

cat("Restricted patient count:", nrow(post_2_restricted_patients), "\n\n")

# ----------------------------
# 2. Download mortality files
# ----------------------------

download_mortality_cycle <- function(file_name, cycle_label) {

  base_url <- "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/"
  url <- paste0(base_url, file_name)

  cat("Downloading:", url, "\n")

  mort <- read_fwf(
    file = url,
    fwf_cols(
      SEQN = c(1, 6),
      ELIGSTAT = c(15, 15),
      MORTSTAT = c(16, 16),
      PERMTH_EXM = c(46, 48)
    ),
    col_types = cols(
      SEQN = col_double(),
      ELIGSTAT = col_double(),
      MORTSTAT = col_double(),
      PERMTH_EXM = col_double()
    )
  ) %>%
    mutate(cycle = cycle_label)

  cat("Rows:", nrow(mort), "\n\n")

  return(mort)
}

mort_g <- download_mortality_cycle(
  "NHANES_2011_2012_MORT_2019_PUBLIC.dat",
  "2011-2012"
)

mort_h <- download_mortality_cycle(
  "NHANES_2013_2014_MORT_2019_PUBLIC.dat",
  "2013-2014"
)

# ----------------------------
# 3. Combine and clean mortality
# ----------------------------

mortality_clean <- bind_rows(mort_g, mort_h) %>%
  transmute(
    SEQN,
    cycle,
    mortality_eligible = if_else(ELIGSTAT == 1, 1, 0),
    death = if_else(MORTSTAT == 1, 1, 0),
    followup_months = PERMTH_EXM,
    followup_years = PERMTH_EXM / 12
  )

cat("Total mortality rows before restriction:", nrow(mortality_clean), "\n")

# ----------------------------
# 4. Keep only restricted patients
# ----------------------------

post_3_restricted_mortality <- post_2_restricted_patients %>%
  left_join(
    mortality_clean,
    by = "SEQN"
  )

cat("Rows after restricting to patient IDs:", nrow(post_3_restricted_mortality), "\n")
cat("Missing mortality rows:", sum(is.na(post_3_restricted_mortality$mortality_eligible)), "\n\n")

# ----------------------------
# 5. Keep mortality eligible only
# ----------------------------

post_3_restricted_mortality_eligible <- post_3_restricted_mortality %>%
  filter(mortality_eligible == 1)

cat("Rows after mortality eligibility filter:", nrow(post_3_restricted_mortality_eligible), "\n\n")

# ----------------------------
# 6. Mortality summary
# ----------------------------

cat("All-cause mortality summary:\n")

mortality_summary <- post_3_restricted_mortality_eligible %>%
  count(death, name = "n") %>%
  mutate(
    percent = round(100 * n / sum(n), 1)
  )

print(as.data.frame(mortality_summary))

# ----------------------------
# 7. Follow-up summary
# ----------------------------

cat("\nFollow-up summary:\n")

followup_summary <- post_3_restricted_mortality_eligible %>%
  summarise(
    n = n(),
    mean_followup_years = round(mean(followup_years, na.rm = TRUE), 2),
    median_followup_years = round(median(followup_years, na.rm = TRUE), 2),
    max_followup_years = round(max(followup_years, na.rm = TRUE), 2)
  )

print(as.data.frame(followup_summary))


Starting restricted all-cause mortality linkage...

Restricted patient count: 4625 

Downloading: https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/NHANES_2011_2012_MORT_2019_PUBLIC.dat 
Rows: 9756 

Downloading: https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/NHANES_2013_2014_MORT_2019_PUBLIC.dat 
Rows: 10175 

Total mortality rows before restriction: 19931 
Rows after restricting to patient IDs: 4625 
Missing mortality rows: 0 

Rows after mortality eligibility filter: 4625 

All-cause mortality summary:
  death    n percent
1     0 3824    82.7
2     1  801    17.3

Follow-up summary:
     n mean_followup_years median_followup_years max_followup_years
1 4625                6.43                  6.58               9.25


In addition: Warning messages:
1: One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat) 
2: One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat) 


In [8]:
%%R

library(dplyr)

cat("Mortality eligibility counts:\n\n")

post_3_restricted_mortality %>%
  count(mortality_eligible, name = "n") %>%
  mutate(
    percent = round(100 * n / sum(n), 1)
  ) %>%
  print()

cat("\nMissing mortality eligibility values:\n")

sum(is.na(post_3_restricted_mortality$mortality_eligible))

Mortality eligibility counts:

  mortality_eligible    n percent
1                  1 4625     100

Missing mortality eligibility values:
[1] 0


In [9]:
%%R

library(dplyr)

cat("Death counts:\n\n")

post_3_restricted_mortality_eligible %>%
  count(death, name = "n") %>%
  mutate(
    percent = round(100 * n / sum(n), 1)
  ) %>%
  print()

cat("\nInterpretation:\n")
cat("death = 0 --> alive at end of follow-up\n")
cat("death = 1 --> died during follow-up\n")

Death counts:

  death    n percent
1     0 3824    82.7
2     1  801    17.3

Interpretation:
death = 0 --> alive at end of follow-up
death = 1 --> died during follow-up


# Save the Covariates

In [10]:
%%R

library(dplyr)

cat("Creating patient-level Post_4_Covariates table with full debug...\n\n")

# ----------------------------
# 0. Show all objects
# ----------------------------

cat("Objects currently available:\n")
print(ls())

cat("\nObject dimensions if available:\n")

object_names <- c(
  "time_fixed_covariates",
  "time_fixed_survival",
  "post_3_restricted_mortality_eligible",
  "post_3_restricted_mortality",
  "post_2_restricted_patients",
  "baseline_table"
)

for (obj in object_names) {
  if (exists(obj)) {
    cat("\n", obj, "exists\n", sep = "")
    print(dim(get(obj)))
    cat("Columns:\n")
    print(names(get(obj)))
  } else {
    cat("\n", obj, "does NOT exist\n", sep = "")
  }
}

# ----------------------------
# 1. Pick covariate data
# ----------------------------

cat("\nPicking covariate dataset...\n")

if (exists("time_fixed_covariates")) {
  covariate_data <- time_fixed_covariates
  cat("Using time_fixed_covariates\n")
} else {
  stop("time_fixed_covariates not found. You need to re-run the code that creates the patient-level covariate dataset.")
}

cat("\nCovariate data dimensions:\n")
print(dim(covariate_data))

cat("\nFirst 5 covariate rows:\n")
print(as.data.frame(head(covariate_data, 5)))

# ----------------------------
# 2. Pick mortality data
# ----------------------------

cat("\nPicking mortality dataset...\n")

if (exists("post_3_restricted_mortality_eligible")) {
  mortality_data <- post_3_restricted_mortality_eligible
  cat("Using post_3_restricted_mortality_eligible\n")
} else if (exists("post_3_restricted_mortality")) {
  mortality_data <- post_3_restricted_mortality
  cat("Using post_3_restricted_mortality\n")
} else {
  stop("No mortality dataset found. Re-run mortality linkage code.")
}

cat("\nMortality data dimensions:\n")
print(dim(mortality_data))

cat("\nFirst 5 mortality rows:\n")
print(as.data.frame(head(mortality_data, 5)))

# ----------------------------
# 3. Check join keys
# ----------------------------

cat("\nChecking join keys...\n")

cat("SEQN in covariate_data:", "SEQN" %in% names(covariate_data), "\n")
cat("cycle in covariate_data:", "cycle" %in% names(covariate_data), "\n")
cat("SEQN in mortality_data:", "SEQN" %in% names(mortality_data), "\n")
cat("cycle in mortality_data:", "cycle" %in% names(mortality_data), "\n")

cat("\nUnique cycles in covariate_data:\n")
print(unique(covariate_data$cycle))

cat("\nUnique cycles in mortality_data:\n")
print(unique(mortality_data$cycle))

cat("\nDuplicate SEQN-cycle rows in covariate_data:\n")
print(
  covariate_data %>%
    count(SEQN, cycle) %>%
    filter(n > 1) %>%
    nrow()
)

cat("\nDuplicate SEQN-cycle rows in mortality_data:\n")
print(
  mortality_data %>%
    count(SEQN, cycle) %>%
    filter(n > 1) %>%
    nrow()
)

# ----------------------------
# 4. Merge covariates with mortality
# ----------------------------

cat("\nMerging covariates with mortality by SEQN and cycle...\n")

merged_data <- covariate_data %>%
  inner_join(
    mortality_data %>%
      select(SEQN, cycle, mortality_eligible, death, followup_months, followup_years),
    by = c("SEQN", "cycle")
  )

cat("Merged data dimensions:\n")
print(dim(merged_data))

cat("\nFirst 5 merged rows:\n")
print(as.data.frame(head(merged_data, 5)))

cat("\nMissing death after merge:\n")
print(sum(is.na(merged_data$death)))

cat("\nMissing followup_months after merge:\n")
print(sum(is.na(merged_data$followup_months)))

# ----------------------------
# 5. Check expected patient-level columns
# ----------------------------

needed_cols <- c(
  "SEQN",
  "cycle",
  "age",
  "bmi",
  "sex",
  "race_ethnicity",
  "education",
  "marital_status",
  "smoking_status",
  "alcohol_use",
  "hypertension",
  "diabetes",
  "heart_attack",
  "stroke",
  "cancer",
  "self_rated_health",
  "mortality_eligible",
  "death",
  "followup_months",
  "followup_years"
)

cat("\nChecking needed columns in merged data...\n")

missing_cols <- setdiff(needed_cols, names(merged_data))

if (length(missing_cols) > 0) {
  cat("Missing columns:\n")
  print(missing_cols)
  stop("Merged data is still missing needed columns.")
} else {
  cat("All needed columns found.\n")
}

# ----------------------------
# 6. Create final patient-level table
# ----------------------------

post_4_covariates <- merged_data %>%
  select(all_of(needed_cols)) %>%
  arrange(SEQN)

cat("\nFinal patient-level Post_4_Covariates dimensions:\n")
print(dim(post_4_covariates))

cat("\nFirst 10 rows of final patient-level table:\n")
print(as.data.frame(head(post_4_covariates, 10)))

cat("\nDeath counts:\n")
print(table(post_4_covariates$death, useNA = "ifany"))

cat("\nMortality eligibility counts:\n")
print(table(post_4_covariates$mortality_eligible, useNA = "ifany"))

cat("\nFollow-up months summary:\n")
print(summary(post_4_covariates$followup_months))

# ----------------------------
# 7. Save final patient-level table
# ----------------------------

write.csv(
  post_4_covariates,
  "/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/Post_4_Covariates.csv",
  row.names = FALSE
)

cat("\nSaved patient-level file:\n")
cat("/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/Post_4_Covariates.csv\n")

Creating patient-level Post_4_Covariates table with full debug...

Objects currently available:
 [1] "age_summary"                         
 [2] "alq_g"                               
 [3] "alq_h"                               
 [4] "alq_raw"                             
 [5] "baseline_table"                      
 [6] "bmi_summary"                         
 [7] "bmx_g"                               
 [8] "bmx_h"                               
 [9] "bmx_raw"                             
[10] "bpq_g"                               
[11] "bpq_h"                               
[12] "bpq_raw"                             
[13] "demo_g"                              
[14] "demo_h"                              
[15] "demo_raw"                            
[16] "diq_g"                               
[17] "diq_h"                               
[18] "diq_raw"                             
[19] "download_mortality_cycle"            
[20] "download_table"                      
[21] "followup_summary" 